# Notebook 05 — Hyperparameter Tuning (XGBoost)

**Project:** Telco Customer Churn Prediction — IIT Roorkee Capstone

This notebook:
1. Explains why we tune XGBoost (the best classical model from CV)
2. Defines the hyperparameter search space
3. Runs `RandomizedSearchCV` (20 iterations × 5-fold = 100 fits)
4. Compares untuned vs tuned CV ROC-AUC
5. Saves the tuned pipeline — overwriting `best_model.joblib`

**Prerequisite:** Run Notebook 03 first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import loguniform, randint
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style='whitegrid', context='talk')

DATA_PATH   = Path('..') / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODEL_PATH  = Path('..') / 'outputs' / 'models' / 'best_model.joblib'
REPORTS_DIR = Path('..') / 'outputs' / 'reports'
FIGURES_DIR = Path('..') / 'outputs' / 'figures'
RANDOM_STATE = 42
print('Ready.')

In [ ]:
df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['customerID'])

y = df['Churn']
X = df.drop(columns=['Churn'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

NUMERICAL_COLUMNS   = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_COLUMNS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
]
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 1. Why Random Search, Not Grid Search?

XGBoost has many hyperparameters with large ranges. A full grid over:
- n_estimators ∈ {100, 200, 300, 400, 500}
- max_depth ∈ {3, 4, 5, 6, 7, 8}
- learning_rate ∈ {0.01, 0.05, 0.1, 0.2, 0.3}
- ... plus 3 more params

would take **hours**. Bergstra & Bengio (2012) showed that **random sampling** over the same space typically finds near-optimal values in a fraction of the time — because not all hyperparameters are equally important, and the search is not wasted on unimportant dimensions.

We use `n_iter=20` → 100 fits (20 × 5-fold). Runs in under 10 minutes.

## 2. Measure Untuned XGBoost (Baseline)

Record the default configuration's CV score so we can measure the lift from tuning.

In [ ]:
from sklearn.model_selection import cross_val_score

# Load from CV scores if available
cv_scores_path = REPORTS_DIR / 'cv_scores.json'
untuned_cv_auc = None
if cv_scores_path.exists():
    with open(cv_scores_path) as f:
        cv_data = json.load(f)
    if 'scores' in cv_data and 'XGBoost' in cv_data['scores']:
        untuned_cv_auc = cv_data['scores']['XGBoost']['mean']
        print(f'Untuned XGBoost CV ROC-AUC (from Notebook 03): {untuned_cv_auc:.4f}')

if untuned_cv_auc is None:
    print('cv_scores.json not found — running CV for untuned XGBoost now...')
    untuned_pipe = Pipeline(steps=[
        ('preprocess', ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), NUMERICAL_COLUMNS),
                ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
            ], remainder='drop')),
        ('model', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                scale_pos_weight=2.77, eval_metric='logloss',
                                tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    scores = cross_val_score(
        untuned_pipe, X_train, y_train,
        scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=1,
    )
    untuned_cv_auc = scores.mean()
    print(f'Untuned XGBoost CV ROC-AUC: {untuned_cv_auc:.4f} (+/- {scores.std():.4f})')

## 3. Define Hyperparameter Search Space

| Parameter | Distribution | Rationale |
|-----------|-------------|-----------|
| `n_estimators` | randint(100, 600) | More trees = better but slower |
| `max_depth` | randint(3, 9) | Controls tree complexity (overfitting risk) |
| `learning_rate` | loguniform(0.01, 0.3) | Log-scale because small values matter |
| `subsample` | {0.7, 0.8, 0.9, 1.0} | Row sampling per tree (regularisation) |
| `colsample_bytree` | {0.6, 0.8, 1.0} | Column sampling per tree (regularisation) |
| `min_child_weight` | randint(1, 8) | Min sum of instance weights in a child leaf |

Note: parameters use the `model__` prefix because they live inside a Pipeline step named `model`.

In [ ]:
PARAM_DIST = {
    'model__n_estimators':     randint(100, 600),
    'model__max_depth':        randint(3, 9),
    'model__learning_rate':    loguniform(0.01, 0.3),
    'model__subsample':        [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__min_child_weight': randint(1, 8),
}
print('Parameter search space defined:')
for k, v in PARAM_DIST.items():
    print(f'  {k}: {v}')

In [ ]:
xgb_pipe = Pipeline(steps=[
    ('preprocess', ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), NUMERICAL_COLUMNS),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
        ], remainder='drop')),
    ('model', XGBClassifier(
        scale_pos_weight=2.77,   # fixed: class imbalance ratio
        eval_metric='logloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
print('XGBoost pipeline ready for tuning.')

## 4. Run RandomizedSearchCV

20 random combinations × 5-fold CV = 100 fits. `refit=True` leaves a fully-fitted pipeline at `search.best_estimator_`.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=PARAM_DIST,
    n_iter=20,
    scoring='roc_auc',
    cv=skf,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,
)

print('Running RandomizedSearchCV (20 iters × 5-fold = 100 fits)...')
search.fit(X_train, y_train)
print(f'\nBest CV ROC-AUC: {search.best_score_:.4f}')
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

## 5. Explore Search Results

In [ ]:
results_df = pd.DataFrame(search.cv_results_)
results_df = results_df.sort_values('rank_test_score')[[
    'rank_test_score', 'mean_test_score', 'std_test_score',
    'param_model__n_estimators', 'param_model__max_depth',
    'param_model__learning_rate', 'param_model__subsample'
]]
print('Top 10 configurations from random search:')
results_df.head(10).round(4)

In [ ]:
# Distribution of all 20 trial scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trial scores ranked
scores_sorted = sorted(results_df['mean_test_score'], reverse=True)
axes[0].bar(range(1, len(scores_sorted)+1), scores_sorted, color='#4C72B0')
axes[0].axhline(untuned_cv_auc, color='red', linestyle='--',
                label=f'Untuned baseline ({untuned_cv_auc:.4f})')
axes[0].set_xlabel('Trial (ranked by score)')
axes[0].set_ylabel('CV ROC-AUC')
axes[0].set_title('All 20 random search trials')
axes[0].legend()

# Right: learning rate vs score
axes[1].scatter(
    results_df['param_model__learning_rate'].astype(float),
    results_df['mean_test_score'],
    alpha=0.7, s=80, color='#DD8452'
)
axes[1].set_xscale('log')
axes[1].set_xlabel('learning_rate (log scale)')
axes[1].set_ylabel('CV ROC-AUC')
axes[1].set_title('Learning rate vs CV score')

plt.tight_layout()
plt.show()

## 6. Before vs After Comparison

In [ ]:
tuned_cv_auc   = search.best_score_
tuned_test_auc = roc_auc_score(y_test, search.best_estimator_.predict_proba(X_test)[:, 1])

untuned_pipe_for_test = Pipeline(steps=[
    ('preprocess', ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), NUMERICAL_COLUMNS),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
        ], remainder='drop')),
    ('model', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                            scale_pos_weight=2.77, eval_metric='logloss',
                            tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)),
])
untuned_pipe_for_test.fit(X_train, y_train)
untuned_test_auc = roc_auc_score(y_test, untuned_pipe_for_test.predict_proba(X_test)[:, 1])

print('=== XGBoost — Before vs After Tuning ===')
print(f'  Untuned CV ROC-AUC:   {untuned_cv_auc:.4f}')
print(f'  Tuned   CV ROC-AUC:   {tuned_cv_auc:.4f}   (+{tuned_cv_auc - untuned_cv_auc:+.4f})')
print(f'  Untuned Test ROC-AUC: {untuned_test_auc:.4f}')
print(f'  Tuned   Test ROC-AUC: {tuned_test_auc:.4f}   (+{tuned_test_auc - untuned_test_auc:+.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
categories = ['CV ROC-AUC', 'Test ROC-AUC']
untuned_vals = [untuned_cv_auc, untuned_test_auc]
tuned_vals   = [tuned_cv_auc,   tuned_test_auc]

x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, untuned_vals, width, label='Untuned XGBoost', color='#4C72B0', alpha=0.8)
ax.bar(x + width/2, tuned_vals,   width, label='Tuned XGBoost',   color='#DD8452', alpha=0.8)

for i, (u, t) in enumerate(zip(untuned_vals, tuned_vals)):
    ax.text(i - width/2, u + 0.001, f'{u:.4f}', ha='center', fontsize=11)
    ax.text(i + width/2, t + 0.001, f'{t:.4f}', ha='center', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0.8, 0.87)
ax.set_ylabel('ROC-AUC')
ax.set_title('XGBoost: Untuned vs Tuned')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Save Tuned Model

The tuned pipeline (preprocessor + tuned XGBoost) replaces `best_model.joblib`. Downstream notebooks (importance, threshold, prediction) load this file automatically.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(search.best_estimator_, MODEL_PATH)
print(f'Saved tuned model to: {MODEL_PATH}')

# Update the CV scores JSON with tuning results
cv_record = {}
if cv_scores_path.exists():
    with open(cv_scores_path) as f:
        cv_record = json.load(f)

cv_record.update({
    'tuned_model': 'XGBoost',
    'tuned_cv_roc_auc': float(tuned_cv_auc),
    'untuned_cv_roc_auc': float(untuned_cv_auc),
    'best_params': {
        k: (v.item() if hasattr(v, 'item') else v)
        for k, v in search.best_params_.items()
    },
})

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
with open(cv_scores_path, 'w') as f:
    json.dump(cv_record, f, indent=2)
print('Updated: outputs/reports/cv_scores.json')

In [ ]:
# Verify the saved model loads and scores correctly
loaded = joblib.load(MODEL_PATH)
loaded_auc = roc_auc_score(y_test, loaded.predict_proba(X_test)[:, 1])
print(f'Verification — loaded model test ROC-AUC: {loaded_auc:.4f}')
print(f'Best params stored in model:')
for k, v in loaded.named_steps['model'].get_params().items():
    if k in ['n_estimators', 'max_depth', 'learning_rate', 'subsample']:
        print(f'  {k}: {v}')

## Summary

- Random search explored 20 XGBoost configurations × 5-fold = 100 fits
- Tuning typically lifts CV ROC-AUC by 0.5–1.5 points over the default configuration
- The tuned pipeline is saved to `outputs/models/best_model.joblib`
- Downstream notebooks automatically use the tuned model

**Next:** [06_feature_importance.ipynb](06_feature_importance.ipynb)